# S45_02 — Tokenizers

Tokenizers convert raw text into integer token IDs that a model can process. The choice of tokenizer is tied to the model — you must use the same tokenizer the model was trained with.

## Tokenization algorithms

| Algorithm | Used by | How it works |
|-----------|---------|-------------|
| **BPE** (Byte-Pair Encoding) | GPT-2, GPT-4, Llama | Start with characters; merge most frequent pairs iteratively |
| **WordPiece** | BERT, DistilBERT | Like BPE but maximises language model likelihood; subwords get `##` prefix |
| **SentencePiece** | T5, Llama, Mistral | Language-agnostic; works on raw bytes; `▁` marks word starts |

In [ ]:
from transformers import AutoTokenizer

# --- BERT tokenizer (WordPiece) ---
bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
text = 'Transformers are revolutionizing NLP'
tokens = bert_tok.tokenize(text)
ids = bert_tok.encode(text)
print('BERT tokens:', tokens)
print('BERT IDs:   ', ids)
# Note: 'revolutionizing' → ['revolution', '##izing'] (WordPiece subwords)

In [ ]:
# --- GPT-2 tokenizer (BPE) ---
gpt_tok = AutoTokenizer.from_pretrained('gpt2')
tokens = gpt_tok.tokenize(text)
print('GPT-2 tokens:', tokens)
# Space before words is part of the token: 'Ġare', 'Ġrevolution'
print(f'GPT-2 vocab size: {gpt_tok.vocab_size:,}')

## Special tokens

Each model uses special tokens for structure. Get confused about these and your fine-tuning will break.

In [ ]:
# BERT special tokens
print('BERT special tokens:')
print(f'  [CLS]: {bert_tok.cls_token_id}')   # prepended to every sequence
print(f'  [SEP]: {bert_tok.sep_token_id}')   # separates segments
print(f'  [PAD]: {bert_tok.pad_token_id}')   # padding
print(f'  [MASK]: {bert_tok.mask_token_id}') # used in MLM pre-training

# Full encoding with special tokens
encoded = bert_tok('Hello world', 'Second sentence', return_tensors='pt')
print('\nFull encoding keys:', list(encoded.keys()))
# input_ids, attention_mask, token_type_ids

In [ ]:
# Batch encoding with padding and truncation
sentences = [
    'Short sentence.',
    'A much longer sentence that has more tokens and needs to be handled carefully.',
    'Medium length text here.',
]

batch = bert_tok(
    sentences,
    padding=True,        # pad to longest in batch
    truncation=True,     # truncate to max_length
    max_length=20,
    return_tensors='pt',
)

print(f'input_ids shape:      {batch["input_ids"].shape}')       # (3, 20)
print(f'attention_mask shape: {batch["attention_mask"].shape}')  # (3, 20)
# attention_mask = 1 for real tokens, 0 for padding

## Token counting — know your costs

LLM API pricing is per-token. Always count tokens before sending to an API.

In [ ]:
# tiktoken is OpenAI's tokenizer (cl100k_base used by GPT-4, gpt-3.5-turbo)
# pip install tiktoken
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')  # GPT-4 encoding
text = 'The transformer architecture revolutionized natural language processing in 2017.'
tokens = enc.encode(text)
print(f'Token count: {len(tokens)}')
print(f'Tokens: {[enc.decode([t]) for t in tokens]}')

**Rule of thumb:** 1 token ≈ 0.75 words (English). 1000 tokens ≈ 750 words ≈ 1.5 pages of text.

Next: [S45_03_datasets_library.ipynb](./S45_03_datasets_library.ipynb)
